# EMPIRE Model — Scenario Comparison (Trinity vs. REPowerEU++ vs. GoRES vs. NECPEssentials)

This notebook compares the European-level results of four EMPIRE macro-scenarios,
each stored in its own result folder (same file structure as produced by
`run_empire.py`, only the folder name changes):

```
Data_handler_energy_vis_Trinity
Data_handler_energy_vis_REPowerEU++
Data_handler_energy_vis_GoRES
Data_handler_energy_vis_NECPEssentials
```

Charts use only the short scenario labels (**Trinity**, **REPowerEU++**, **GoRES**,
**NECPEssentials**) — never the full folder path.

Comparisons included, chosen to highlight what actually differs between energy
scenarios of this kind:

1. Installed capacity mix, final period — side-by-side technology bars
2. Generation mix, final period — side-by-side technology bars + RES share
3. Capacity trajectory over time, per scenario (build-out speed)
4. CO₂ emissions trajectory & CO₂ price trajectory
5. Total system cost (objective function) comparison
6. Electricity price level and dispersion across nodes
7. Storage capacity (power & energy) — final period
8. Transmission capacity — final period + growth
9. Curtailment of renewables — final period
10. Consolidated KPI comparison table


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from io import StringIO

pd.set_option("display.max_columns", 50)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## 1. Configuration

`BASE_PREFIX` is the common folder-name stem; each scenario only changes the
suffix after it. Edit `BASE_DIR` if the folders live somewhere other than the
current working directory.

In [ ]:
BASE_DIR = Path(".")           # <-- CHANGE ME if folders are elsewhere  r_energy_vis_
BASE_PREFIX = "dataset_Data_handler_"
SCENARIOS = {
    "Trinity":        BASE_DIR / f"{BASE_PREFIX}Trinity",
    "REPowerEU++":    BASE_DIR / f"{BASE_PREFIX}REPowerEU++",
    "GoRES":          BASE_DIR / f"{BASE_PREFIX}GoRES",
    "NECPEssentials": BASE_DIR / f"{BASE_PREFIX}NECPEssentials",
}

SCEN_COLORS = {
    "Trinity": "#2e75b6",
    "REPowerEU++": "#c0392b",
    "GoRES": "#4c9a2a",
    "NECPEssentials": "#e07b39",
}

for name, path in SCENARIOS.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:16s} -> {path}  [{status}]")


## 2. Loading helpers (multi-block parsers + per-scenario loader)

In [ ]:
def load_csv(result_dir, filename, **kwargs):
    fp = result_dir / filename
    if not fp.exists():
        return None
    return pd.read_csv(fp, **kwargs)


def read_europeplot_blocks(path):
    '''Parse the multi-block EuropePlot CSV into a dict of {block_title: DataFrame}.'''
    if not path.exists():
        return {}
    with open(path) as f:
        raw_lines = [l.rstrip("\n") for l in f]

    blocks = {}
    i = 0
    current_title = None
    buffer = []

    def flush():
        nonlocal buffer, current_title
        if current_title and buffer:
            txt = "\n".join(buffer)
            df = pd.read_csv(StringIO(txt))
            df = df.rename(columns={df.columns[0]: "Technology"})
            blocks[current_title] = df
        buffer = []

    while i < len(raw_lines):
        line = raw_lines[i]
        if line.strip() == "":
            i += 1
            continue
        cells_ = line.split(",")
        looks_like_title = (len(cells_) == 2 and cells_[0] == "Period" and cells_[1] != "")
        if looks_like_title:
            flush()
            current_title = cells_[1]
            i += 1
            header = raw_lines[i]
            buffer = [header]
        else:
            buffer.append(line)
        i += 1
    flush()
    return blocks


def read_europesummary(path):
    if not path.exists():
        return None, None, None
    with open(path) as f:
        lines = [l.rstrip("\n") for l in f]
    blank_idx = [k for k, l in enumerate(lines) if l.strip() == ""]
    if not blank_idx:
        return pd.read_csv(StringIO("\n".join(lines))), None, None

    b1_txt = "\n".join(lines[:blank_idx[0]])
    summary_df = pd.read_csv(StringIO(b1_txt))

    gen_totals_df, stor_totals_df = None, None
    if len(blank_idx) >= 2:
        b2_txt = "\n".join(lines[blank_idx[0]+1:blank_idx[1]])
        gen_totals_df = pd.read_csv(StringIO(b2_txt))
        remainder = [l for l in lines[blank_idx[1]+1:] if l.strip() != ""]
        if remainder:
            stor_totals_df = pd.read_csv(StringIO("\n".join(remainder)))
    return summary_df, gen_totals_df, stor_totals_df


def load_scenario(result_dir):
    '''Load every file needed for the comparison from one scenario's result folder.'''
    data = {}
    data["gen"] = load_csv(result_dir, "results_output_gen.csv")
    data["stor"] = load_csv(result_dir, "results_output_stor.csv")
    data["trans"] = load_csv(result_dir, "results_output_transmision.csv")
    data["curt"] = load_csv(result_dir, "results_output_curtailed_prod.csv")
    data["blocks"] = read_europeplot_blocks(result_dir / "results_output_EuropePlot.csv")
    data["summary"], data["gen_totals"], data["stor_totals"] = read_europesummary(
        result_dir / "results_output_EuropeSummary.csv"
    )
    obj_path = result_dir / "results_objective.csv"
    if obj_path.exists():
        with open(obj_path) as f:
            data["objective"] = float(f.readline().strip().split(":")[-1])
    else:
        data["objective"] = None
    return data

SCEN_DATA = {name: load_scenario(path) for name, path in SCENARIOS.items()}


## 3. Common helpers for cross-scenario charts

In [ ]:
def final_period_series(block_df):
    '''Given a Technology x Period EuropePlot block, return the Series for the last period.'''
    if block_df is None:
        return None
    d = block_df.set_index(block_df.columns[0])
    last_col = d.columns[-1]
    return pd.to_numeric(d[last_col], errors="coerce").fillna(0)


def grouped_bar_by_tech(series_dict, title, ylabel, top_n=None):
    '''series_dict: {scenario_name: pd.Series indexed by technology}. Draws grouped bars.'''
    df = pd.DataFrame(series_dict).fillna(0)
    df = df.loc[(df.abs().sum(axis=1) > 1e-6)]
    if top_n:
        df = df.loc[df.sum(axis=1).sort_values(ascending=False).index[:top_n]]
    else:
        df = df.loc[df.sum(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(len(df.index))
    n = len(df.columns)
    width = 0.8 / n
    for i, scen in enumerate(df.columns):
        ax.bar(x + i * width - 0.4 + width/2, df[scen], width=width,
               label=scen, color=SCEN_COLORS.get(scen))
    ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=40, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    plt.tight_layout()
    return fig, ax


## 4. Installed capacity mix — final period, by scenario

Shows how the four scenarios differ in the technology portfolio they end up
building.

In [ ]:
cap_series = {}
for name, data in SCEN_DATA.items():
    s = final_period_series(data["blocks"].get("genInstalledCap_MW"))
    if s is None and data["gen"] is not None:
        last_period = data["gen"]["Period"].iloc[-1]
        s = data["gen"][data["gen"]["Period"] == last_period].groupby("GeneratorType")["genInstalledCap_MW"].sum()
    cap_series[name] = s

grouped_bar_by_tech(cap_series, "Installed Generation Capacity, Final Period", "MW")
plt.show()

totals = {name: (s.sum() if s is not None else np.nan) for name, s in cap_series.items()}
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(totals.keys(), totals.values(), color=[SCEN_COLORS[k] for k in totals])
ax.set_ylabel("MW")
ax.set_title("Total Installed Capacity, Final Period")
plt.tight_layout(); plt.show()


## 5. Generation mix — final period, by scenario + RES share

RES share is a key differentiator between "green" scenarios (e.g. GoRES) and
more conservative ones (e.g. NECPEssentials).

In [ ]:
RES_TECHS = ["Windonshore", "Windoffshore", "Solar", "Hydrorun-of-the-river",
             "Hydroregulated", "Wave", "Bio", "BioCCS"]

gen_series = {}
for name, data in SCEN_DATA.items():
    s = final_period_series(data["blocks"].get("genExpectedAnnualProduction_GWh"))
    if s is None and data["gen"] is not None:
        last_period = data["gen"]["Period"].iloc[-1]
        s = data["gen"][data["gen"]["Period"] == last_period].groupby("GeneratorType")["genExpectedAnnualProduction_GWh"].sum()
    gen_series[name] = s

grouped_bar_by_tech(gen_series, "Electricity Generation, Final Period", "GWh/yr")
plt.show()

res_share = {}
for name, s in gen_series.items():
    if s is None:
        res_share[name] = np.nan
        continue
    res = s[[t for t in RES_TECHS if t in s.index]].sum()
    res_share[name] = 100 * res / s.sum() if s.sum() else np.nan

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(res_share.keys(), res_share.values(), color=[SCEN_COLORS[k] for k in res_share])
ax.set_ylabel("% of annual generation")
ax.set_title("Renewable Energy Share, Final Period")
for i, (k, v) in enumerate(res_share.items()):
    if not np.isnan(v):
        ax.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.tight_layout(); plt.show()


## 6. Capacity build-out trajectory over time

Compares *how fast* each scenario builds new capacity (total across all
technologies, per period), rather than only the end-state.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for name, data in SCEN_DATA.items():
    block = data["blocks"].get("genInstalledCap_MW")
    if block is None:
        continue
    d = block.set_index(block.columns[0]).apply(pd.to_numeric, errors="coerce").fillna(0)
    totals_over_time = d.sum(axis=0)
    ax.plot(totals_over_time.index, totals_over_time.values, marker="o",
            label=name, color=SCEN_COLORS.get(name))
ax.set_ylabel("MW")
ax.set_title("Total Installed Generation Capacity Over Time")
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout(); plt.show()


## 7. CO₂ emissions & CO₂ price trajectories

Emissions trajectories reveal the decarbonisation pace; the CO₂ price/shadow
price shows how binding the carbon constraint is in each scenario.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for name, data in SCEN_DATA.items():
    summary = data["summary"]
    if summary is None:
        continue
    per_period = summary.groupby("Period").agg(
        AnnualCO2emission_Mt=("AnnualCO2emission_Ton", lambda x: x.mean() / 1e6),
        CO2Price_EuroPerTon=("CO2Price_EuroPerTon", "mean"),
    ).reset_index()
    axes[0].plot(per_period["Period"], per_period["AnnualCO2emission_Mt"], marker="o",
                 label=name, color=SCEN_COLORS.get(name))
    axes[1].plot(per_period["Period"], per_period["CO2Price_EuroPerTon"], marker="o",
                 label=name, color=SCEN_COLORS.get(name))

axes[0].set_ylabel("Mt CO2/yr")
axes[0].set_title("CO2 Emissions Over Time")
axes[0].legend()
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

axes[1].set_ylabel("EUR/ton CO2")
axes[1].set_title("CO2 Price Over Time")
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout(); plt.show()


## 8. Total system cost (objective function)

The discounted total system cost is the single most direct "which scenario
is cheapest" comparison.

In [ ]:
obj_values = {name: data["objective"] for name, data in SCEN_DATA.items()}
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(obj_values.keys(), obj_values.values(), color=[SCEN_COLORS[k] for k in obj_values])
ax.set_ylabel("EUR (discounted objective value)")
ax.set_title("Total System Cost by Scenario")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
for b, (k, v) in zip(bars, obj_values.items()):
    if v is not None:
        ax.text(b.get_x() + b.get_width()/2, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()


## 9. Electricity price level & dispersion across nodes

Average European price by period per scenario, plus a boxplot of node-level
average prices in the final period (dispersion = a proxy for how integrated /
congested the grid is in each scenario).

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for name, data in SCEN_DATA.items():
    summary = data["summary"]
    if summary is None:
        continue
    price_by_period = summary.groupby("Period")["AvgELPrice_EuroPerMWh"].mean()
    ax.plot(price_by_period.index, price_by_period.values, marker="o",
            label=name, color=SCEN_COLORS.get(name))
ax.set_ylabel("EUR/MWh")
ax.set_title("European Average Electricity Price Over Time")
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout(); plt.show()

# Node-level price dispersion in the final period, only if the full hourly
# operational file exists for a scenario (skipped otherwise, e.g. OUT_OF_SAMPLE runs)
box_data, box_labels = [], []
for name in SCENARIOS:
    result_dir = SCENARIOS[name]
    op_path = result_dir / "results_output_Operational.csv"
    if not op_path.exists():
        print(f"[skip] {name}: no results_output_Operational.csv")
        continue
    cols = pd.read_csv(op_path, nrows=0).columns.tolist()
    needed = [c for c in ["Node", "Period", "Price_EURperMWh"] if c in cols]
    op = pd.read_csv(op_path, usecols=needed)
    last_period = op["Period"].iloc[-1]
    node_avg = op[op["Period"] == last_period].groupby("Node")["Price_EURperMWh"].mean()
    box_data.append(node_avg.values)
    box_labels.append(name)

if box_data:
    fig, ax = plt.subplots(figsize=(7, 5))
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
    for patch, label in zip(bp["boxes"], box_labels):
        patch.set_facecolor(SCEN_COLORS.get(label, "#888888"))
        patch.set_alpha(0.6)
    ax.set_ylabel("EUR/MWh")
    ax.set_title("Node-Level Average Price Dispersion, Final Period")
    plt.tight_layout(); plt.show()


## 10. Storage capacity — final period, by scenario

Compares how much flexibility (power, MW) and duration (energy, MWh) each
scenario relies on.

In [ ]:
storPW_series, storEN_series = {}, {}
for name, data in SCEN_DATA.items():
    s_pw = final_period_series(data["blocks"].get("storPWInstalledCap_MW"))
    s_en = final_period_series(data["blocks"].get("storENInstalledCap_MW"))
    if s_pw is None and data["stor"] is not None:
        last_period = data["stor"]["Period"].iloc[-1]
        s_pw = data["stor"][data["stor"]["Period"] == last_period].groupby("StorageType")["storPWInstalledCap_MW"].sum()
        s_en = data["stor"][data["stor"]["Period"] == last_period].groupby("StorageType")["storENInstalledCap_MWh"].sum()
    storPW_series[name] = s_pw
    storEN_series[name] = s_en

grouped_bar_by_tech(storPW_series, "Storage Power Capacity, Final Period", "MW")
plt.show()
grouped_bar_by_tech(storEN_series, "Storage Energy Capacity, Final Period", "MWh")
plt.show()


## 11. Transmission capacity — final period and growth

Total cross-border capacity is a proxy for how much each scenario relies on
inter-regional exchange rather than domestic build-out.

In [ ]:
trans_totals_final, trans_growth = {}, {}
for name, data in SCEN_DATA.items():
    trans = data["trans"]
    if trans is None:
        continue
    last_period = trans["Period"].iloc[-1]
    trans_totals_final[name] = trans[trans["Period"] == last_period]["transmissionInstalledCap_MW"].sum()
    trans_growth[name] = trans.groupby("Period")["transmissionInstalledCap_MW"].sum()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(trans_totals_final.keys(), trans_totals_final.values(),
       color=[SCEN_COLORS[k] for k in trans_totals_final])
ax.set_ylabel("MW")
ax.set_title("Total Cross-Border Transmission Capacity, Final Period")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for name, series in trans_growth.items():
    ax.plot(series.index, series.values, marker="o", label=name, color=SCEN_COLORS.get(name))
ax.set_ylabel("MW")
ax.set_title("Total Transmission Capacity Over Time")
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout(); plt.show()


## 12. Curtailment of renewables — final period

Higher curtailment can indicate insufficient flexibility (storage/transmission)
relative to the RES build-out in a given scenario.

In [ ]:
curt_series = {}
for name, data in SCEN_DATA.items():
    curt = data["curt"]
    if curt is None:
        continue
    last_period = curt["Period"].iloc[-1]
    curt_series[name] = curt[curt["Period"] == last_period].groupby("RESGeneratorType")["ExpectedAnnualCurtailment_GWh"].sum()

if curt_series:
    grouped_bar_by_tech(curt_series, "Curtailed RES Generation, Final Period", "GWh/yr")
    plt.show()

    totals = {name: (s.sum() if s is not None else np.nan) for name, s in curt_series.items()}
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(totals.keys(), totals.values(), color=[SCEN_COLORS[k] for k in totals])
    ax.set_ylabel("GWh/yr")
    ax.set_title("Total RES Curtailment, Final Period")
    plt.tight_layout(); plt.show()


## 13. Consolidated KPI comparison table

One row per scenario — the numbers to quote directly in the report.

In [ ]:
rows = []
for name, data in SCEN_DATA.items():
    row = {"Scenario": name}
    row["Total system cost (EUR)"] = data["objective"]

    cap_s = cap_series.get(name)
    row["Installed capacity, final period (MW)"] = cap_s.sum() if cap_s is not None else np.nan

    gen_s = gen_series.get(name)
    row["Generation, final period (GWh)"] = gen_s.sum() if gen_s is not None else np.nan
    row["RES share, final period (%)"] = res_share.get(name)

    summary = data["summary"]
    if summary is not None:
        last_period = summary["Period"].iloc[-1]
        last = summary[summary["Period"] == last_period]
        row["CO2 emissions, final period (Mt)"] = last["AnnualCO2emission_Ton"].mean() / 1e6
        row["Avg. electricity price, final period (EUR/MWh)"] = last["AvgELPrice_EuroPerMWh"].mean()
        row["Curtailed RES, final period (GWh)"] = last["TotAnnualCurtailedRES_GWh"].mean()

    row["Storage power cap., final period (MW)"] = (storPW_series.get(name).sum()
                                                      if storPW_series.get(name) is not None else np.nan)
    row["Storage energy cap., final period (MWh)"] = (storEN_series.get(name).sum()
                                                        if storEN_series.get(name) is not None else np.nan)
    row["Transmission cap., final period (MW)"] = trans_totals_final.get(name, np.nan)

    rows.append(row)

kpi_comparison_df = pd.DataFrame(rows).set_index("Scenario").round(1)
kpi_comparison_df


---
### Notes for the report

- All values labelled "final period" refer to the last investment period
  present in each scenario's own result files — make sure all four scenarios
  were run with the same set of investment periods before comparing directly.
- The grouped bar charts intentionally sort technologies by combined size
  across scenarios, so the largest differences are easiest to spot on the left.
- If a scenario is missing the full hourly `results_output_Operational.csv`
  (e.g. it was solved as `OUT_OF_SAMPLE`), the price-dispersion boxplot simply
  skips it — check the printed status message.
- To add a fifth scenario, just add one more entry to the `SCENARIOS` dict in
  Section 1; every chart below iterates over that dict automatically.
